# Fit and infer with the coordination-age JK+ model

**Audience:** project collaborators fitting or deploying the finalized coordination-age model.

**Prerequisites:** install `requirements.txt` and run this notebook from anywhere inside the repository. The checked-in `Data/X.csv` and `Data/y.csv` already satisfy the approved-feature, blacklist, duplicate-removal, `n ≥ 5`, and timing-validity rules.

**Outcome:** fit and save 144 leave-one-out KRR pipelines plus one full-data production pipeline, then reload them and produce a point prediction with a 95% jackknife+ interval.


## Outline

1. Locate the project and validate `X` and `y`.
2. Run the fitting script and save all artifacts under `artifacts/jackknife_plus_krr/`.
3. Report LOOCV out-of-fold RMSE, MAE, and R².
4. Reload the saved objects and infer one example subject at 95% requested marginal coverage.
5. Reuse the same function for a genuinely new 32-feature row.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts' / 'fit_jackknife_plus_krr.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the coordination_age project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

X_PATH = PROJECT_ROOT / 'Data' / 'X.csv'
Y_PATH = PROJECT_ROOT / 'Data' / 'y.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'jackknife_plus_krr'
REFIT = False  # Set True to rebuild all 145 models; inference needs no refitting.
PROJECT_ROOT


## 1. Load and validate the frozen model inputs

`X.csv` contains exactly the selected 32 predictors. `y.csv` contains only `AgeInYears`; row order aligns the two files. Missing predictor values are expected and are median-imputed inside each training fold.


In [ ]:
X = pd.read_csv(X_PATH)
y_table = pd.read_csv(Y_PATH)

assert X.shape == (144, 32)
assert y_table.columns.tolist() == ['AgeInYears']
assert len(y_table) == len(X)
assert X.columns.is_unique

pd.Series({
    'subjects': len(X),
    'features': X.shape[1],
    'missing_feature_cells': int(X.isna().sum().sum()),
    'minimum_age_years': float(y_table['AgeInYears'].min()),
    'maximum_age_years': float(y_table['AgeInYears'].max()),
})


## 2. Use the saved models or explicitly refit

The bundled artifacts are ready for inference. By default, this cell skips
fitting. Set `REFIT=True` in the setup cell to tune and fit 144 leave-one-out
pipelines and one full-data pipeline using five-fold inner CV.


In [ ]:
fit_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'fit_jackknife_plus_krr.py'),
    '--x-csv', str(X_PATH),
    '--y-csv', str(Y_PATH),
    '--target-column', 'AgeInYears',
    '--output-dir', str(ARTIFACT_DIR),
    '--inner-folds', '5',
    '--n-jobs', '6',
    '--progress-every', '5',
]
if REFIT:
    subprocess.run(fit_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Using the bundled artifacts. Set REFIT=True above to rebuild them.")


In [ ]:
manifest_path = ARTIFACT_DIR / 'coordination_age_krr_jackknife_plus_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pd.Series({
    'artifact_directory': str(ARTIFACT_DIR),
    'loo_models': manifest['n_loo_models'],
    'production_models': manifest['n_production_models'],
    'total_models': manifest['total_models'],
})


## 3. LOOCV out-of-fold performance

Each prediction below comes from the corresponding leave-one-subject-out pipeline, including five-fold inner-CV KRR tuning. These are the full-cohort OOF diagnostics; the residuals are also the calibration scores used by JK+.


In [ ]:
residuals = pd.read_csv(ARTIFACT_DIR / 'coordination_age_krr_jackknife_plus_residuals.csv')
observed = residuals['observed_y'].to_numpy(dtype=float)
predicted = residuals['loo_prediction'].to_numpy(dtype=float)
errors = observed - predicted

oof_metrics = pd.Series({
    'LOOCV OOF RMSE (years)': float(np.sqrt(np.mean(errors ** 2))),
    'LOOCV OOF MAE (years)': float(np.mean(np.abs(errors))),
    'LOOCV OOF R²': float(1.0 - np.sum(errors ** 2) / np.sum((observed - observed.mean()) ** 2)),
}, name='value')
oof_metrics


## 4. Reload artifacts and demonstrate 95% JK+ inference

The full-data model supplies the point prediction. The 144 leave-one-out models and their aligned absolute residuals supply the jackknife+ endpoints. Here the first stored row is used only as a schema-correct demonstration; replace it with a genuinely new subject for deployment.


In [ ]:
from scripts.fit_jackknife_plus_krr import (
    LOO_MODELS_FILENAME,
    MANIFEST_FILENAME,
    PRODUCTION_MODEL_FILENAME,
    RESIDUALS_FILENAME,
)
from scripts.infer_jackknife_plus_krr import (
    load_jackknife_plus_artifacts,
    predict_with_jackknife_plus,
)

loo_artifact, production_model, residuals = load_jackknife_plus_artifacts(
    loo_models_path=ARTIFACT_DIR / LOO_MODELS_FILENAME,
    production_model_path=ARTIFACT_DIR / PRODUCTION_MODEL_FILENAME,
    residuals_path=ARTIFACT_DIR / RESIDUALS_FILENAME,
    manifest_path=ARTIFACT_DIR / MANIFEST_FILENAME,
)

example_X = X.iloc[[0]].copy()
example_prediction = predict_with_jackknife_plus(
    example_X,
    loo_artifact=loo_artifact,
    production_model=production_model,
    residuals=residuals,
    alpha=0.05,
)
example_prediction[['prediction', 'jkplus_lower', 'jkplus_upper', 'jkplus_width', 'requested_coverage']]


## 5. Infer a new subject

Provide a DataFrame with the same 32 feature names in the same order. Values must already obey the upstream domain rules; missing values may remain as `NaN` for pipeline imputation.


In [ ]:
def infer_new_subjects(new_X: pd.DataFrame, coverage: float = 0.95) -> pd.DataFrame:
    expected = list(loo_artifact['feature_columns'])
    if new_X.columns.tolist() != expected:
        raise ValueError('new_X must contain the exact 32 training columns in order.')
    if not 0 < coverage < 1:
        raise ValueError('coverage must lie strictly between 0 and 1.')
    return predict_with_jackknife_plus(
        new_X,
        loo_artifact=loo_artifact,
        production_model=production_model,
        residuals=residuals,
        alpha=1.0 - coverage,
    )

# Exercise: replace this with pd.read_csv('path/to/new_X.csv').
infer_new_subjects(example_X, coverage=0.95)


## Operational cautions

- Pass only the 32 approved predictors; do not include subject identifiers or age.
- Do not reorder or rename predictors.
- Blacklist enforcement, copied-trial removal, aggregation, `n ≥ 5` masking, and approved-feature selection happen before these model scripts.
- Keep the entire artifact directory together: model/residual alignment is checked through the manifest and SHA-256 digests.
- The 95% statement is finite-sample marginal JK+ coverage under exchangeability, not conditional coverage for every feature profile.
